# 0910 — Expressed protein mass of the six ordered plasmids

Computes the molecular weight of the protein each pET-28a(+) construct actually
expresses, for reading today's SDS-PAGE gel.

**Answer first — the gel has three resolvable bands, not six:**

| Band | Construct(s) | Mass |
|---|---|---|
| ~92 kDa | `CDS10_WT_NOTAG` | 91.6 kDa |
| ~72 kDa | `CDS11_WT_CTERM_HIS6TAG` | 72.0 kDa |
| ~42 kDa | `CDS12_WT_NOTAG` **and all three** `CDS12_ANC_Node*` | 41.8–42.3 kDa |

The four CDS_0012-family constructs (WT + three reconstructed ancestors) all have
1086 bp inserts and land within **0.43 kDa (~1%)** of each other. **SDS-PAGE cannot
tell them apart** — not from each other and not from WT. Distinguishing the
ancestral variants needs sequencing or intact-mass MS, not a gel.

## Two traps this notebook handles

1. **The ORF wraps around the origin.** Twist rotated each plasmid so position 1 is
   the *insert*. That puts pET-28a's T7 start and its N-terminal His6 leader at the
   very **end** of the file. Translating from position 1 silently drops the leader
   and gives a mass ~2.3 kDa too low. The code finds the leader motif and rotates.
2. **The filenames lie.** "NOTAG" constructs are N-terminally His6-tagged — they
   inherit pET-28a's `MGSSHHHHHHSSGLVPRGSHM` leader (21 aa, +2294.5 Da to the chain). This is
   already documented in
   [`../documents/2026-07-27_twist_construct_order/README.md`](../documents/2026-07-27_twist_construct_order/README.md);
   this notebook re-derives it independently and asserts the two agree.

## Sources

| Set | Path | Format |
|---|---|---|
| 1 (2026-07-27) | `documents/2026-07-27_twist_construct_order/constructs/*.gb` | 3 GenBank files |
| 2 (filed 2026-09-10) | `documents/2026-09-10_twist_construct_order_second_set/constructs/iGEM-design_Second_Set.csv` | `Construct sequence` column |

In [1]:
import re
from pathlib import Path

import pandas as pd
from Bio.Seq import Seq
from Bio.SeqUtils.ProtParam import ProteinAnalysis

ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
SET1 = ROOT / "documents/2026-07-27_twist_construct_order/constructs"
SET2 = ROOT / "documents/2026-09-10_twist_construct_order_second_set/constructs"
OUT  = ROOT / "code/output/0910_construct_protein_mw"
OUT.mkdir(parents=True, exist_ok=True)

for p in (SET1, SET2):
    assert p.is_dir(), p

## 1. The pET-28a(+) leader

Everything hangs off this motif. It runs from the T7 initiator ATG (inside the
`CCATGG` NcoI site) through to the `CATATG` NdeI site where the insert starts —
so its final `ATG` is shared with NdeI. 63 nt = 21 codons.

In [2]:
LEADER_DNA = "ATGGGCAGCAGCCATCATCATCATCATCACAGCAGCGGCCTGGTGCCGCGCGGCAGCCATATG"
LEADER_AA  = "MGSSHHHHHHSSGLVPRGSHM"

assert str(Seq(LEADER_DNA).translate()) == LEADER_AA
print(f"{LEADER_AA}   {len(LEADER_AA)} aa, {ProteinAnalysis(LEADER_AA).molecular_weight():.1f} Da as a free peptide")

MGSSHHHHHHSSGLVPRGSHM   21 aa, 2312.5 Da as a free peptide


## 2. Parse and translate

`orf_from_circular` rotates the circular plasmid so the ORF starts at the leader,
then translates to the first in-frame stop.

The three `assert`s are the point of this cell. An internal stop, a leader that
does not translate to `MGSSHHHHHHSSGLVPRGSH...`, or more than one motif match all
mean the frame or the rotation broke — better to fail loudly here than to hand a
wrong kDa to someone standing at the gel.

In [3]:
def read_gb_origin(path: Path) -> str:
    'Twist .gb files use a non-standard FEATURES table, so read the ORIGIN block directly.'
    block = path.read_text().split("ORIGIN", 1)[1].split("//", 1)[0]
    return "".join(re.findall(r"[acgtACGT]", block)).upper()


def orf_from_circular(dna: str, name: str) -> str:
    'Rotate a circular plasmid to the T7 initiator ATG and translate to the first stop.'
    hits = [m.start() for m in re.finditer(LEADER_DNA, dna + dna[: len(LEADER_DNA)])]
    assert len(hits) == 1, f"{name}: expected exactly 1 leader match, got {len(hits)}"

    rot  = (dna + dna)[hits[0] : hits[0] + len(dna)]      # ORF may wrap the origin
    prot = str(Seq(rot).translate())
    stop = prot.find("*")
    assert stop != -1, f"{name}: no in-frame stop codon"

    aa = prot[:stop]
    assert aa.startswith(LEADER_AA), f"{name}: leader is {aa[:21]!r}, not {LEADER_AA!r}"
    assert "*" not in aa,            f"{name}: internal stop codon -> frame is wrong"
    return aa

In [4]:
def masses(aa: str) -> dict:
    pa  = ProteinAnalysis(aa)
    mat = ProteinAnalysis(aa[1:])   # E. coli MAP cleaves the initiator Met (residue 2 is Gly)
    _, eps_red = pa.molar_extinction_coefficient()   # (cystines, all-reduced)
    mw = pa.molecular_weight()
    return dict(
        length_aa        = len(aa),
        mw_da            = mw,
        mw_kda           = mw / 1000,
        mw_kda_minus_met = mat.molecular_weight() / 1000,
        pI               = pa.isoelectric_point(),
        n_cys            = aa.count("C"),
        eps_reduced      = eps_red,
        a280_1mg_per_ml  = eps_red / mw,
    )

In [5]:
rows, seqs = [], {}

# --- set 1: three .gb ---
for f in sorted(SET1.glob("*.gb")):
    dna = read_gb_origin(f)
    aa  = orf_from_circular(dna, f.name)
    seqs[f.stem] = aa
    rows.append(dict(name=f.stem, source="set 1 (.gb)", plasmid_bp=len(dna),
                     insert_aa=len(aa) - len(LEADER_AA), **masses(aa)))

# --- set 2: Twist design sheet ---
csv = pd.read_csv(SET2 / "iGEM-design_Second_Set.csv")
for _, r in csv.iterrows():
    dna = r["Construct sequence"].strip().upper()
    assert len(dna) == r["Construct length"],  f"{r['Name']}: length disagrees with the sheet"
    assert dna.endswith("GGCAGCCATATG"),       f"{r['Name']}: does not end at the NdeI site"
    aa = orf_from_circular(dna, r["Name"])
    seqs[r["Name"]] = aa
    rows.append(dict(name=r["Name"], source="set 2 (csv)", plasmid_bp=len(dna),
                     insert_aa=len(aa) - len(LEADER_AA), **masses(aa)))

df = pd.DataFrame(rows).sort_values("mw_kda", ascending=False).reset_index(drop=True)
print(f"{len(df)} constructs, all assertions passed")

6 constructs, all assertions passed


## 3. Cross-check against the filed record

`documents/2026-07-27_twist_construct_order/README.md` records the residue ranges
the PI ordered. These are derived completely independently (that README was
written from a manual translation), so agreement is real validation, not a
tautology.

- `CDS12_WT` — CDS_0012 res 117–477 → **361** residues
- `CDS10_WT` — CDS_0010 res 105–917 → **813** residues
- `CDS11_WT` — CDS_0011 res 305–941 (**637**) + a designed 30 aa
  `GGSGGGGS`–His6–`ENLYFQ`–`GGGGSGGGGS` cassette → **667** residues

In [6]:
for name, expect in [("CDS12_WT_NOTAG", 361), ("CDS10_WT_NOTAG", 813), ("CDS11_WT_CTERM_HIS6TAG", 667)]:
    got = len(seqs[name]) - len(LEADER_AA)
    print(f"{name:24s} insert {got:4d} aa   expect {expect:4d}   {'OK' if got == expect else 'MISMATCH'}")

print("\nCDS11 N-terminus (vector leader + designed cassette):")
print(" ", seqs["CDS11_WT_CTERM_HIS6TAG"][:52])
print("  " + "^" * 21 + " " * 0 + "  vector leader (His6 #1)")

CDS12_WT_NOTAG           insert  361 aa   expect  361   OK
CDS10_WT_NOTAG           insert  813 aa   expect  813   OK
CDS11_WT_CTERM_HIS6TAG   insert  667 aa   expect  667   OK

CDS11 N-terminus (vector leader + designed cassette):
  MGSSHHHHHHSSGLVPRGSHMGGSGGGGSHHHHHHENLYFQGGGGSGGGGSD
  ^^^^^^^^^^^^^^^^^^^^^  vector leader (His6 #1)


## 4. The table

In [7]:
cols = ["name", "source", "plasmid_bp", "length_aa", "insert_aa",
        "mw_da", "mw_kda", "mw_kda_minus_met", "pI", "n_cys", "eps_reduced", "a280_1mg_per_ml"]
df[cols].round({"mw_da": 1, "mw_kda": 2, "mw_kda_minus_met": 2, "pI": 2, "a280_1mg_per_ml": 3})

,name,source,plasmid_bp,length_aa,insert_aa,mw_da,mw_kda,mw_kda_minus_met,pI,n_cys,eps_reduced,a280_1mg_per_ml
0,CDS10_WT_NOTAG,set 1 (.gb),7734,834,813,91632.7,91.63,91.50,6.82,5,219920,2.400
1,CDS11_WT_CTERM_HIS6TAG,set 1 (.gb),7296,688,667,72020.5,72.02,71.89,6.65,2,67855,0.942
2,CDS12_ANC_Node281,set 2 (csv),6378,382,361,42268.5,42.27,42.14,5.82,0,119860,2.836
3,CDS12_ANC_Node40,set 2 (csv),6378,382,361,42199.1,42.20,42.07,5.30,0,125360,2.971
4,CDS12_ANC_Node1,set 2 (csv),6378,382,361,41907.3,41.91,41.78,6.34,1,111380,2.658
5,CDS12_WT_NOTAG,set 1 (.gb),6378,382,361,41835.4,41.84,41.70,7.25,0,112870,2.698


## 5. What to expect on the gel

In [8]:
print("Expected bands (reduced, denatured monomer)\n" + "-" * 60)
for kda, grp in df.groupby(df.mw_kda.round(0), sort=False):
    who = ", ".join(grp.name)
    print(f"  ~{kda:>3.0f} kDa   {who}")

spread = df.query("insert_aa == 361").mw_kda
print(f"\nCDS_0012 family spread: {spread.min():.2f}-{spread.max():.2f} kDa "
      f"= {1000*(spread.max()-spread.min()):.0f} Da "
      f"({100*(spread.max()-spread.min())/spread.mean():.1f}% of the mass) -> one band, not four.")

Expected bands (reduced, denatured monomer)
------------------------------------------------------------
  ~ 92 kDa   CDS10_WT_NOTAG
  ~ 72 kDa   CDS11_WT_CTERM_HIS6TAG
  ~ 42 kDa   CDS12_ANC_Node281, CDS12_ANC_Node40, CDS12_ANC_Node1, CDS12_WT_NOTAG

CDS_0012 family spread: 41.84-42.27 kDa = 433 Da (1.0% of the mass) -> one band, not four.


### Reading the gel — caveats in the order they will bite

1. **Initiator Met.** *E. coli* MAP cleaves it here (residue 2 is Gly), costing
   131 Da. Invisible on a gel; matters only for intact-mass MS. The
   `mw_kda_minus_met` column is the mature mass.
2. **Ni-NTA works on every construct.** All six carry the vector's N-terminal His6;
   CDS11 carries a second, designed His6 as well. None has a C-terminal tag —
   every insert stops before the vector's downstream His6. **Do not run a
   C-terminal-tag detection step.** (PI confirmed 2026-07-29.)
3. **CDS11 is a phage tail fiber.** Fibers migrate anomalously high and are often
   SDS-resistant: a trimer can survive boiling and run near 3 × 72 = ~216 kDa.
   A band well above 72 kDa is a plausible trimer, not necessarily a contaminant.
4. **Anomalous migration generally.** Acidic and Pro-rich proteins run 5–20% high.
   Calculated mass is a prediction, not a ruler.
5. **The ANC inserts are `Codon Optimized: No`** and use mammalian-like codon bias
   (`CCC`/`GGG`/`GCC`-heavy), not *E. coli*. If an expected ~42 kDa band is missing
   from an ANC lane but present for WT, rare-codon stalling is the first suspect —
   try Rosetta/BL21-CodonPlus before concluding the protein is insoluble.
6. **A280 → concentration.** Divide A280 by `a280_1mg_per_ml` for mg/mL. Note
   CDS11's is only 0.94 — it has just **5 Trp in 688 aa** (0.7%, vs 3.7% and 4.5% for CDS10 and CDS12), so A280 is a weak readout for it; use
   BCA instead.

## 6. Save

In [9]:
tsv = OUT / "construct_protein_mw.tsv"
df[cols].to_csv(tsv, sep="\t", index=False, float_format="%.4f")

fasta = OUT / "expressed_proteins.faa"
with fasta.open("w") as fh:
    for _, r in df.iterrows():
        aa = seqs[r["name"]]
        fh.write(f">{r['name']} {len(aa)}aa {r['mw_kda']:.2f}kDa pI={r['pI']:.2f}\n")
        for i in range(0, len(aa), 60):
            fh.write(aa[i:i + 60] + "\n")

print(tsv.relative_to(ROOT))
print(fasta.relative_to(ROOT))

code/output/0910_construct_protein_mw/construct_protein_mw.tsv
code/output/0910_construct_protein_mw/expressed_proteins.faa


---
*Log — 2026-09-10: written while an SDS-PAGE gel of these constructs was staining.
Six constructs across the two Twist orders (3 + 3); **this gel ran only the three
`CDS12_ANC_Node*` from set 2** (confirmed 2026-09-10). All three sit at ~42 kDa within
0.36 kDa of each other, so the gel reads as one band position in three lanes — with no
`CDS12_WT` lane, it cannot be used to compare ancestors against the extant sequence. Masses re-derived from sequence and
cross-checked against the residue ranges in
`documents/2026-07-27_twist_construct_order/README.md`; all three agree.
The headline result is that the four CDS_0012-family constructs are one band, not four.*